# Checks — Workshop I — Ingestion

Self-study helper for the static Workshop Guide and Databricks Free Edition. In class the Workshop Guide app runs the same checks with its **Check** button.

Load it in `WORKSHOP_ingestion` with `%run ./_checks/ws_ingestion` after the setup cells. It defines `check(workshop, step)`, which reads the tables your step cells save (`<catalog>.default._ws_ws_ingestion_<step>`) and your Bronze table, prints PASS or FAIL and records the result.

> **Note** Do not edit this notebook. Add a cell `check("ws_ingestion", "tNN")` under a step and run it.

In [ ]:
%run ./_progress

In [ ]:
# Shared helpers: failures carry a participant-facing message
from pyspark.sql import functions as _F


class _CheckFail(Exception):
    """Raised by a step check; the message is shown to the participant."""


def _expect(condition, message):
    if not condition:
        raise _CheckFail(message)


def _need(name, where):
    value = globals().get(name)
    _expect(value is not None and value is not ..., f"`{name}` is not defined yet. Run {where} first.")
    return value

In [ ]:
# Saved results and schema helpers
def _saved(step, where):
    """Table saved by the last line of a step cell: <catalog>.default._ws_ws_ingestion_<step>."""
    name = f"{_need('CATALOG', '00_setup')}.default._ws_ws_ingestion_{step}"
    try:
        df = spark.table(name)
        _ = df.columns
    except Exception:
        raise _CheckFail(f"Table _ws_ws_ingestion_{step} not found. Run {where} to the end: its last line saves your result.")
    return df


def _types(df):
    return {f.name: f.dataType.simpleString() for f in df.schema.fields}


def _has_cols(df, name, cols):
    missing = [c for c in cols if c not in df.columns]
    _expect(not missing, f"`{name}` is missing column(s) {missing}. Check the names and the `.alias(...)` / `AS` parts.")

In [ ]:
# Steps with two DataFrames save both in one table with a `_part` column
def _saved_part(step, part, name, where):
    """Rows of one part, with the columns that part filled (union fills the others with NULL)."""
    df = _saved(step, where)
    _expect("_part" in df.columns, f"The saved result has no `_part` column. Run {where} again with its last line unchanged.")
    df = df.filter(_F.col("_part") == part).drop("_part")
    filled = df.select([_F.count(_F.col(f"`{c}`")).alias(c) for c in df.columns]).first()
    _expect(df.count() > 0, f"`{name}` is empty or was not saved. Run {where} again, including the last line.")
    return df.select([f"`{c}`" for c in df.columns if filled[c] > 0])

In [ ]:
# Step registry used by check()
_STEPS = {}


def _step(step_id, step_type):
    def register(fn):
        _STEPS[step_id] = (step_type, fn)
        return fn
    return register

In [ ]:
# t02 — CSV with an inferred schema
@_step("t02", "order")
def _t02():
    df = _saved("t02", "the step 02 cell")
    _expect("customer_id" in df.columns and "email" in df.columns,
            "Columns are named `_c0`, `_c1`, ...: the reader did not use the first row as the header. Check the `header` option.")
    _expect(len(df.columns) == 10, f"Expected 10 columns, got {len(df.columns)}. Check the format and the options.")
    n = df.count()
    _expect(n > 0, "`customers_infer_df` is empty. Check the path passed to `.load(...)`.")
    return f"customers_infer_df loaded: {n:,} rows, {len(df.columns)} columns."

In [ ]:
# t03 — CSV with an explicit DDL schema
_T03_COLS = ["customer_id", "first_name", "last_name", "email", "phone",
             "city", "state", "country", "registration_date", "customer_segment"]


@_step("t03", "fill")
def _t03():
    df = _saved("t03", "the step 03 cell")
    _expect(df.columns == _T03_COLS, f"Column names or order differ from the CSV. Got: {df.columns}")
    t = _types(df)
    wrong = [c for c in _T03_COLS if t[c] != ("date" if c == "registration_date" else "string")]
    _expect(not wrong, f"Wrong type for {wrong}: registration_date must be DATE, the other columns STRING.")
    n = df.count()
    _expect(n > 0, "`customers_df` is empty. Check the path and the `header` option.")
    with_at = df.filter(_F.col("email").contains("@")).count()
    _expect(with_at > 0.9 * n, "Most `email` values have no '@': the schema columns are probably in the wrong order.")
    return f"explicit schema OK, {n:,} rows."

In [ ]:
# t04 (optional) — explore the customer DataFrame (total_rows and distinct_countries saved as one row)
@_step("t04", "code")
def _t04():
    base = _saved("t03", "the step 03 cell")
    df = _saved("t04", "your step 04 cell")
    _has_cols(df, "step 04 result", ["total_rows", "distinct_countries"])
    rows = df.select("total_rows", "distinct_countries").collect()
    _expect(len(rows) == 1, "The saved step 04 result must be one row. Run the step 04 cell again.")
    total, countries = rows[0]["total_rows"], rows[0]["distinct_countries"]
    _expect(total == base.count(), f"`total_rows` is {total:,}, but customers_df has {base.count():,} rows.")
    expected = base.select("country").distinct().count()
    _expect(countries == expected, f"`distinct_countries` is {countries}; count the distinct values of `country` instead.")
    return f"{total:,} rows, {countries} distinct country, summary computed."

In [ ]:
# t05 — JSON orders with a schema
@_step("t05", "fill")
def _t05():
    df = _saved("t05", "the step 05 cell")
    _has_cols(df, "orders_df", ["order_id", "customer_id", "order_datetime", "total_amount", "payment_method"])
    t = _types(df)
    _expect(t["total_amount"] == "double", "`total_amount` must be DOUBLE: pass `orders_schema` to the reader.")
    _expect(t["order_datetime"] == "string", "`order_datetime` must stay a STRING in this step (you cast it in step 07).")
    n = df.count()
    _expect(n > 0, "`orders_df` is empty. Check the format and the path.")
    ord_ok = df.filter(_F.col("order_id").startswith("ORD")).count()
    _expect(ord_ok > 0.9 * n,
            "`order_id` values do not look like ORD...: the file was not parsed as JSON. Check the format name.")
    amount_ok = df.filter(_F.col("total_amount").isNotNull()).count()
    _expect(amount_ok > 0.9 * n,
            f"Only {amount_ok:,} of {n:,} rows have `total_amount`. Check the format name and the schema field names.")
    return f"orders loaded: {n:,} rows."

In [ ]:
# t06 (optional) — filter customers (`_part` ny_ca / recent)
@_step("t06", "code")
def _t06():
    base, where = _saved("t03", "the step 03 cell"), "your step 06 cell"
    ny_ca = _saved_part("t06", "ny_ca", "customers_ny_ca", where)
    states = {r["state"] for r in ny_ca.select("state").distinct().collect()}
    _expect(states == {"NY", "CA"}, f"`customers_ny_ca` must contain the states NY and CA only, got {sorted(map(str, states))}.")
    expected = base.filter(_F.col("state").isin("NY", "CA")).count()
    _expect(ny_ca.count() == expected, f"`customers_ny_ca` has {ny_ca.count():,} rows, expected {expected:,}. Do not drop or duplicate rows.")
    recent = _saved_part("t06", "recent", "customers_recent", where)
    _has_cols(recent, "customers_recent", ["registration_date"])
    old = recent.filter(_F.year("registration_date") < 2025).count()
    _expect(old == 0, f"`customers_recent` contains {old:,} customers registered before 2025. Compare the year with >= 2025.")
    expected = base.filter(_F.year("registration_date") >= 2025).count()
    _expect(recent.count() == expected, f"`customers_recent` has {recent.count():,} rows, expected {expected:,}.")
    return f"NY/CA: {ny_ca.count():,} rows, registered 2025+: {recent.count():,} rows."

In [ ]:
# t07 — new columns and cast (customers part, `_part` customers)
def _t07_customers():
    df = _saved_part("t07", "customers", "customers_enriched", "the step 07 cell")
    _has_cols(df, "customers_enriched", ["first_name", "last_name", "full_name", "registration_year"])
    year_type = _types(df)["registration_year"]
    _expect(year_type == "int", f"`registration_year` must be INT, got {year_type.upper()}. Use the year function on registration_date.")
    both = _F.col("first_name").isNotNull() & _F.col("last_name").isNotNull()
    good = _F.concat_ws(" ", "first_name", "last_name")
    bad = df.filter(both & ~_F.col("full_name").eqNullSafe(good)).count()
    _expect(bad == 0, f"{bad:,} rows have a wrong `full_name`. Expected first name, one space, last name.")

In [ ]:
# t07 — new columns and cast (orders part, `_part` orders)
@_step("t07", "fill")
def _t07():
    _t07_customers()
    orders = _saved("t05", "the step 05 cell")
    df = _saved_part("t07", "orders", "orders_typed", "the step 07 cell")
    ts_type = _types(df).get("order_ts")
    _expect(ts_type == "timestamp", f"`order_ts` must be TIMESTAMP, got {ts_type}. Cast `order_datetime` to timestamp.")
    src = orders.filter(_F.col("order_datetime").isNotNull()).count()
    dst = df.filter(_F.col("order_ts").isNotNull()).count()
    _expect(src == dst, f"{src - dst:,} non-NULL `order_datetime` values became NULL in `order_ts`.")
    return "full_name, registration_year and order_ts OK."

In [ ]:
# t08 — conditional column customer_tier
@_step("t08", "fill")
def _t08():
    df = _saved("t08", "the step 08 cell")
    _has_cols(df, "customers_tiered", ["customer_tier", "registration_date"])
    nulls = df.filter(_F.col("customer_tier").isNull()).count()
    _expect(nulls == 0, f"`customer_tier` has {nulls:,} NULL values. Add the final `.otherwise(...)` / ELSE branch.")
    year = _F.year("registration_date")
    good = _F.when(year >= 2025, "New").when(year >= 2024, "Regular").otherwise("Loyal")
    bad = df.filter(~_F.col("customer_tier").eqNullSafe(good)).count()
    _expect(bad == 0, f"{bad:,} rows have the wrong tier. Check the thresholds and that the >= 2025 branch comes first.")
    tiers = sorted(r["customer_tier"] for r in df.select("customer_tier").distinct().collect())
    return f"tiers {tiers}, every row correct."

In [ ]:
# t09 — aggregation by payment method (the sort order is graded in the guide: a table keeps no row order)
@_step("t09", "order")
def _t09():
    orders = _saved("t05", "the step 05 cell")
    df = _saved("t09", "the step 09 cell")
    _has_cols(df, "payment_stats", ["payment_method", "order_count", "total_revenue", "avg_order_value"])
    rows = df.collect()
    methods = {r["payment_method"] for r in rows}
    expected = {r["payment_method"] for r in orders.select("payment_method").distinct().collect()}
    _expect(methods == expected and len(rows) == len(expected),
            "`payment_stats` must have exactly one row per payment method (NULL included). Group by `payment_method`.")
    _expect(sum(r["order_count"] for r in rows) == orders.count(), "`order_count` values do not add up to all orders. Count rows with COUNT(*).")
    return f"{len(rows)} payment methods aggregated."

In [ ]:
# t10 (optional) — SQL on temp views: the 5 largest states (compared as values, a table keeps no row order)
@_step("t10", "code")
def _t10():
    base = _saved("t03", "the step 03 cell")
    df = _saved("t10", "your step 10 cell")
    _has_cols(df, "top_states", ["state", "customer_count"])
    counts = sorted((r["customer_count"] for r in df.collect()), reverse=True)
    _expect(len(counts) == 5, f"Expected 5 rows, got {len(counts)}. Limit the result to the top 5.")
    top = [r["count"] for r in base.groupBy("state").count().orderBy(_F.desc("count")).limit(5).collect()]
    _expect(counts == top, "The counts are not the 5 largest per state. Group by `state`, count all rows and sort by `customer_count` descending before LIMIT.")
    return f"top 5 states, counts {counts}."

In [ ]:
# t11 — Bronze Delta table
@_step("t11", "fill")
def _t11():
    raw = _saved("t03", "the step 03 cell")
    expected = sorted(raw.columns + ["_load_ts", "_source_file"])
    target = f"{_need('CATALOG', '00_setup')}.{_need('BRONZE_SCHEMA', '00_setup')}.lab_ingestion_customers"
    try:
        table = spark.table(target)
        _ = table.columns
    except Exception:
        raise _CheckFail(f"Table {target} not found. Run the write part of the step 11 cell.")
    _expect(sorted(table.columns) == expected,
            f"The table must be the raw customers_df plus `_load_ts` and `_source_file`, nothing else. Got {table.columns}")
    _expect(_types(table)["_load_ts"] == "timestamp", "`_load_ts` must be TIMESTAMP: use the current timestamp function.")
    return _t11_rows(raw, table, target)

In [ ]:
# t11 — Bronze row count and source path
def _t11_rows(raw, table, target):
    n, expected = table.count(), raw.count()
    _expect(n == expected, f"Bronze keeps every source row: expected {expected:,}, the table has {n:,}. Use mode overwrite, no filters.")
    paths = [r["_source_file"] for r in table.select("_source_file").distinct().limit(5).collect()]
    _expect(any(p and "customers.csv" in p for p in paths),
            "`_source_file` does not contain the path of customers.csv. Take it from the hidden `_metadata` column.")
    return f"{n:,} rows written to {target}."

In [ ]:
# t12 (optional) — join orders with customers (`_part` enriched / by_country; a table keeps no row order)
@_step("t12", "code")
def _t12():
    orders, customers = _saved("t05", "the step 05 cell"), _saved("t03", "the step 03 cell")
    where = "your step 12 cell"
    df = _saved_part("t12", "enriched", "orders_enriched", where)
    _has_cols(df, "orders_enriched", ["order_id", "customer_id", "first_name", "last_name", "country", "total_amount", "payment_method"])
    expected = orders.join(customers, "customer_id", "left").count()
    _expect(df.count() == expected,
            f"`orders_enriched` has {df.count():,} rows, a left join gives {expected:,}. Use a left join on customer_id, no filters.")
    stats = _saved_part("t12", "by_country", "orders_by_country", where)
    _has_cols(stats, "orders_by_country", ["country", "order_count", "total_revenue"])
    return f"{df.count():,} enriched orders (more than {orders.count():,}: duplicate customer IDs), {stats.count()} countries."

In [ ]:
# check(): run one step, print PASS / FAIL, record the result
def check(workshop, step):
    """Run the hidden checks for one workshop step."""
    if workshop != "ws_ingestion" or step not in _STEPS:
        _result(False, f'unknown step {workshop}/{step}. Use check("ws_ingestion", "t02") ... check("ws_ingestion", "t12").')
        return
    step_type, fn = _STEPS[step]
    try:
        ok, message = True, fn()
    except _CheckFail as e:
        ok, message = False, str(e)
    except Exception as e:  # Spark or Python error while inspecting the participant's objects
        first = (str(e).strip().splitlines() or [type(e).__name__])[0][:140]
        ok, message = False, f"{type(e).__name__}: {first}. Re-run your step cell, then this check."
    _result(ok, f"Step {step[1:]}: {message}")
    record(workshop, step, step_type, ok, message[:200])